In [167]:
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [245]:
# This is the code for a subsetted sqanti table
accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900']
col=['isoform', 'chrom', 'strand', 'length', 'exons', 'structural_category',
       'associated_gene', 'associated_transcript', 'ref_length', 'ref_exons',
       'diff_to_TSS', 'diff_to_TTS', 'diff_to_gene_TSS', 'diff_to_gene_TTS',
       'subcategory', 'RTS_stage', 'all_canonical',
       'bite', 'FSM_class','ORF_seq']
span = 2

for acc in tqdm(accessions):

    # prepare sqanti output
    sqanti = pd.read_csv(f"output/{acc}/sqanti/{acc}-hodgepodgemerged_classification.txt", sep="\t",dtype={'ref_length': 'Int64', 'ref_exons': 'Int64','diff_to_TSS': 'Int64', 'diff_to_TTS': 'Int64',
                              'diff_to_gene_TSS': 'Int64', 'diff_to_gene_TTS': 'Int64'})
    #sqanti.fillna("NaN", inplace=True)
    sqanti = sqanti[col]
    
    # prepare read support
    reads = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_tama/{acc}.collapsed.tama.v2_read_support.txt", sep="\t")
    
    # also add gene read count?
    #r_sub = reads[["merge_trans_id","trans_read_count","gene_read_count"]]
    #r_sub.columns = ["isoform", "isoform_reads", "gene_reads"]
    r_sub = reads[["merge_trans_id","trans_read_count"]]
    r_sub.columns = ["isoform", "isoform_reads"]
    
    # add accession ID and merge all information
    acc_df = pd.merge(left=sqanti, right=r_sub, how='left', left_on='isoform', right_on='isoform')
    acc_df["hodgepodge_gene"]=acc_df["associated_gene"].transform(lambda x: ["_".join(x.split("_")[i:i+span]) for i in range(0, len(x.split("_")), span)])
    acc_df=acc_df.explode("hodgepodge_gene")
    acc_df.sort_values(by="hodgepodge_gene", inplace=True)
    
    ### prepare gene type (TODOc)
    nlr = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/nlr_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    nlr["type"]="nlr"
    busco = pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/busco_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    busco["type"]="busco"
    house=pd.read_csv(f"/ebio/abt6_projects7/dliso/dlis/leon_pipeline/difflines_SQANTI/input/ref_sets/new_ids/housekeeping_{acc}.txt", sep="\t", names=["hodgepodge_gene"])
    house["type"]="house"
    all_sets=pd.concat([nlr, busco, house])
    all_sets
    
    # add gene type
    merged_all=pd.merge(left=acc_df, right=all_sets, how='left', left_on='hodgepodge_gene', right_on='hodgepodge_gene')
    
    merged_all['accession'] = acc
    merged_all=merged_all.loc[merged_all["hodgepodge_gene"].str.split("_").str[0]==acc]
    for c in merged_all.columns:
        if c in ['ref_length', 'ref_exons','diff_to_TSS', 'diff_to_TTS', 'diff_to_gene_TSS', 'diff_to_gene_TTS','ORF_seq']:
            merged_all[c] = merged_all[c].astype(str).replace("<NA>","NaN")
    merged_all["type"].fillna("other", inplace=True)
    
    merged_all.sort_values("hodgepodge_gene")

    # merged_all.to_csv(f"/tmp/global2/lvaness/dl20-annex/final_isoseq/sqanti/classification/{acc}-hodgepodgemerged_classification_hodgepodge-subset.txt", sep="\t", index=False)
    merged_all.to_csv(f"output/01/{acc}-hodgepodgemerged_classification_hodgepodge-subset.txt", sep="\t", index=False)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17/17 [02:33<00:00,  9.04s/it]


In [276]:
gene=pd.read_csv("output/01/58090.txt", sep="\t", names=["ORF","reads"])

In [274]:
gene["orf_rs"]=gene.groupby("ORF")["reads"].apply("sum")

In [259]:
gene=gene[["ORF","orf_rs"]].drop_duplicates()

In [260]:
gene=gene.dropna()

In [261]:
read=gene["orf_rs"].sum()

In [ ]:
read

In [266]:
r=np.array(gene["orf_rs"].tolist())

In [269]:
r/read

array([0.01515152, 0.03030303, 0.54545455, 0.03030303, 0.03030303,
       0.04545455, 0.01515152, 0.01515152, 0.01515152, 0.03030303,
       0.01515152, 0.01515152, 0.07575758, 0.03030303, 0.01515152,
       0.07575758])

In [270]:
from scipy.stats import entropy

In [271]:
entropy(r/read)

1.836194596869384

In [244]:
merged_all

,isoform,chrom,strand,length,exons,structural_category,associated_gene,associated_transcript,ref_length,ref_exons,diff_to_TSS,diff_to_TTS,diff_to_gene_TSS,diff_to_gene_TTS,subcategory,RTS_stage,all_canonical,bite,FSM_class,isoform_reads,hodgepodge_gene,type,accession
2104,G1.1,at9900_1_chr1,+,1542,6,full-splice_match,at9900_G000010,at9900_G000010_mRNA01,1685,6,0,143,0,-143,alternative_3end,False,canonical,False,C,1,at9900_G000010,other,at9900
2105,G1.2,at9900_1_chr1,+,1685,6,full-splice_match,at9900_G000010,at9900_G000010_mRNA01,1685,6,0,0,0,0,reference_match,False,canonical,False,C,1,at9900_G000010,other,at9900
2106,G2.2,at9900_1_chr1,-,920,8,novel_in_catalog,at9900_G000020,novel,1652,10,NaN,NaN,24,-92,intron_retention,False,canonical,False,B,1,at9900_G000020,house,at9900
2107,G2.1,at9900_1_chr1,-,1176,7,novel_in_catalog,at9900_G000020,novel,1652,10,NaN,NaN,40,-67,intron_retention,False,canonical,False,B,1,at9900_G000020,house,at9900
2108,G3.1,at9900_1_chr1,-,3742,2,full-splice_match,at9900_G000030,at9900_G000030_mRNA01,3747,2,5,0,5,0,reference_match,False,canonical,False,C,1,at9900_G000030,other,at9900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198817,G16753.5,at9900_9_u178,+,1602,3,full-splice_match,at9900_G337650,at9900_G337650_mRNA03,1637,3,5,30,5,-30,reference_match,False,canonical,False,C,1,at9900_G337650,busco,at9900
198818,G16753.6,at9900_9_u178,+,1152,5,full-splice_match,at9900_G337650,at9900_G337650_mRNA01,1302,5,120,30,120,-30,alternative_5end,False,canonical,False,C,1,at9900_G337650,busco,at9900
198819,G16754.1,at9900_9_u178,-,1301,3,full-splice_match,at9900_G337660,at9900_G337660_mRNA01,1340,3,39,0,39,0,reference_match,False,canonical,False,C,1,at9900_G337660,other,at9900
198820,G16754.3,at9900_9_u178,-,865,4,full-splice_match,at9900_G337660,at9900_G337660_mRNA02,883,4,0,18,0,-18,reference_match,False,canonical,False,C,1,at9900_G337660,other,at9900


In [88]:
accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900']

df = pd.DataFrame()
for acc in tqdm(accessions):

    # -------------1-------------
    # prepare sqanti output
    sqanti = pd.read_csv("output/{}/sqanti/{}-hodgepodgemerged_classification.txt".format(acc,acc), sep="\t")
    sqanti["gene"] = sqanti["isoform"].str.split(".").str[0]
    
    # prepare read support
    reads = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_tama/{}.collapsed.tama.v2_read_support.txt".format(acc), sep="\t")
    # also add gene read count?
    #r_sub = reads[["merge_trans_id","trans_read_count","gene_read_count"]]
    #r_sub.columns = ["isoform", "isoform_reads", "gene_reads"]
    r_sub = reads[["merge_trans_id","trans_read_count"]]
    r_sub.columns = ["isoform", "isoform_reads"]

    # -------------4-------------
    # add accession ID and merge all information
    merged_all = pd.merge(left=sqanti, right=r_sub, how='left', left_on='isoform', right_on='isoform')
    merged_all['accession'] = acc
    merged_all.to_csv(f"/tmp/global2/lvaness/dl20-annex/final_isoseq/sqanti/{acc}-hodgepodgemerged_classification.txt",sep="\t", index=False)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17/17 [02:24<00:00,  8.50s/it]


## Subset the SQANTI table

In [89]:
accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900']

df = pd.DataFrame()
for acc in tqdm(accessions):

    # -------------1-------------
    # prepare sqanti output
    sqanti = pd.read_csv("output/{}/sqanti/{}-hodgepodgemerged_classification.txt".format(acc,acc), sep="\t")
    sub = sqanti[["isoform","exons","length","structural_category","associated_gene","associated_transcript","subcategory","ORF_seq","all_canonical"]]
    sub=sub.copy()
    sub["gene"] = sub["isoform"].str.split(".").str[0]
    
    # prepare read support
    reads = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_tama/{}.collapsed.tama.v2_read_support.txt".format(acc), sep="\t")
    r_sub = reads[["merge_trans_id","trans_read_count","gene_read_count"]]
    r_sub.columns = ["isoform", "isoform_reads", "gene_reads"]

    # -------------4-------------
    # add accession ID and merge all information
    merged_all = pd.merge(left=sub, right=r_sub, how='left', left_on='isoform', right_on='isoform')
    
    merged_all['accession'] = acc

    df = pd.concat([df, merged_all])

df = df[["gene","isoform","accession","associated_gene","associated_transcript","exons","length","isoform_reads","gene_reads","structural_category","subcategory","ORF_seq","all_canonical"]]
df.columns = ["gene","isoform","accession","associated_gene","associated_transcript","exons","length","isoform_read_support","gene_read_support","structural_category","subcategory","Sqanti_ORF_seq","all_canonical"]

df.to_pickle("output/01/all_sqanti.pkl")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17/17 [00:57<00:00,  3.38s/it]


In [90]:
df

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,subcategory,Sqanti_ORF_seq,all_canonical
0,G1,G1.1,at6923,at6923_G000010,at6923_G000010_mRNA01,6,1663,1,2,full-splice_match,reference_match,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSYD...,canonical
1,G1,G1.2,at6923,at6923_G000010,at6923_G000010_mRNA01,6,1588,1,2,full-splice_match,alternative_5end,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSYD...,canonical
2,G10,G10.1,at6923,at6923_G000100,at6923_G000100_mRNA01,3,1640,3,182,full-splice_match,reference_match,MATAFAPTKLTATVPLHGSQENRLLLPIRLAPPSSFLGSTRSLSLR...,canonical
3,G10,G10.10,at6923,at6923_G000100,at6923_G000100_mRNA02,2,1175,1,182,full-splice_match,alternative_3end5end,MCAQMYYRGKMFGFVHLYNGQEAVSTGFIKLLTKSDSVVSTYRDHV...,canonical
4,G10,G10.11,at6923,at6923_G000100,at6923_G000100_mRNA01,3,1473,2,182,full-splice_match,alternative_3end5end,MATAFAPTKLTATVPLHGSQENRLLLPIRLAPPSSFLGSTRSLSLR...,canonical
...,...,...,...,...,...,...,...,...,...,...,...,...,...
197115,G16753,G16753.5,at9900,at9900_G337650,at9900_G337650_mRNA03,3,1602,1,34,full-splice_match,reference_match,MEAKPLNKPTPINLALHTYWNLHSHNSGNILSHKIQLLAGKITPVD...,canonical
197116,G16753,G16753.6,at9900,at9900_G337650,at9900_G337650_mRNA01,5,1152,1,34,full-splice_match,alternative_5end,MVLLCSVNYSLCEKIKTYKLTRGSLSVTFTNYGAVMTSLLLPDRHG...,canonical
197117,G16754,G16754.1,at9900,at9900_G337660,at9900_G337660_mRNA01,3,1301,1,7,full-splice_match,reference_match,MVLVLALGDLHVPHRAADLPPKFKSMLVPGKIQHIICTGNLCIKEI...,canonical
197118,G16754,G16754.2,at9900,at9900_G337660,at9900_G337660_mRNA02,4,847,5,7,full-splice_match,reference_match,MVLVLALGDLHVPHRAADLPPKFKSMLVPGKIQHIICTGNLCIKEI...,canonical


In [91]:
# load full df
ref=pd.read_pickle("output/all_sqanti.pkl")
# add new column
ref["cat_assign"]="None"
# and fill new colum based on 4 categories
ref.loc[(ref["structural_category"] != "antisense") & (ref["structural_category"] != "fusion") & (ref["structural_category"] != "intergenic"), "cat_assign"]="one2one"
ref.loc[(ref["structural_category"] == "antisense"), "cat_assign"]="antisense"
ref.loc[(ref["structural_category"] == "fusion"), "cat_assign"]="fusion"
ref.loc[((ref["structural_category"] == "intergenic") | (ref["structural_category"] == "genic_intron")), "cat_assign"]="novel"

# in some cases, fusion transcripts were reported as one2one if they had perfect splice junctions over their fusion genes.
# filter them and add replace the assignment
#ref["association"] = ref["associated_gene"].str.split("_").str.len()
span = 2
ref["association"]=ref["associated_gene"].transform(lambda x: ["_".join(x.split("_")[i:i+span]) for i in range(0, len(x.split("_")), span)]).str.len()
ref.loc[(ref["cat_assign"]=="one2one")&(ref["association"]>1), "cat_assign"] = "fusion"

In [93]:
def augustus_refs(df):
    out_df= pd.DataFrame()
    accessions=df["accession"].unique().tolist()
    for acc in accessions:    
        busco = pd.read_csv(f"input/ref_sets/new_ids/busco_{acc}.txt", header=None)
        house = pd.read_csv(f"input/ref_sets/new_ids/housekeeping_{acc}.txt", header=None)
        nlr = pd.read_csv(f"input/ref_sets/new_ids/nlrs_v2_{acc}.txt", header=None)

        busco["type"] = "busco"
        house["type"] = "housekeeping"
        nlr["type"] = "nlr"
        gene_type = pd.concat([busco,house,nlr])
        gene_type.columns = ["ref_isoform","type"]
        gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

        gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
        gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"

        sub=df.loc[df["accession"]==acc]
        sub = pd.merge(left=sub, right=gene_type, how='left', left_on='associated_gene', right_on='ref_gene')

        
        out_df=pd.concat([out_df,sub])
    return out_df

In [94]:
def augustus_refs_fusion(df):
    out_df= pd.DataFrame()
    accessions=df["accession"].unique().tolist()
    for acc in accessions:    
        busco = pd.read_csv(f"input/ref_sets/new_ids/busco_{acc}.txt", header=None)
        house = pd.read_csv(f"input/ref_sets/new_ids/housekeeping_{acc}.txt", header=None)
        nlr = pd.read_csv(f"input/ref_sets/new_ids/nlrs_v2_{acc}.txt", header=None)

        busco["type"] = "busco"
        house["type"] = "housekeeping"
        nlr["type"] = "nlr"
        
        gene_type = pd.concat([busco,house,nlr])
        gene_type.columns = ["ref_isoform","type"]
        gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

        gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
        gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
        gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"

        sub=df.loc[df["accession"]==acc]
        sub = pd.merge(left=sub, right=gene_type, how='left', left_on='gene_split', right_on='ref_gene')
        
        out_df=pd.concat([out_df,sub])
    return out_df

In [95]:
fus=ref.loc[ref["cat_assign"]=="fusion"].copy()
fus["gene_split"]=fus["associated_gene"].str.split("_")

span = 2
fus["gene_split"]=fus["gene_split"].transform(lambda x: ["_".join(x[i:i+span]) for i in range(0, len(x), span)])

In [96]:
split=fus.explode("gene_split")

In [97]:
# test for novel genes in fusion genes
for i in split["gene_split"].tolist():
    if i == "novelGene":
        print(i)

In [98]:
fusion_types=augustus_refs_fusion(split).drop(columns=["ref_isoform","ref_gene"])
fusion_types["type"] = fusion_types["type"].fillna("other")

In [99]:
fusion_types.reset_index(drop=True, inplace=True)

In [100]:
fusion_out=pd.DataFrame()
for i in fusion_types.groupby(by=["accession", "gene", "isoform"], as_index=False):
    t=i[1]["type"].value_counts()
    t=t.reset_index()

    t=t["index"].tolist()

    i[1]["fusion_order"] = "_".join(i[1]["type"].tolist())
    
    if "busco" in t:
        i[1]["busco"] = 1
    if "nlr" in t:
        i[1]["nlr"] = 1
    if "housekeeping" in t:
        i[1]["housekeeping"] = 1
    if "other" in t:
        i[1]["other"] = 1

    fusion_out=pd.concat([fusion_out,i[1]], axis=0)

In [101]:
fusion_types= fusion_out.drop(columns=["gene_split","type","association"]).drop_duplicates()
fusion_types.fillna({'other':0, 'busco':0, 'nlr':0,'housekeeping':0}, inplace=True)

## Connect all ref gene sets (Clear Cases + Fusion genes (Order: NLR>House>Busco>Other)) - This creates the DF for subsequent analysis

### The reason to do it this preciesely for the reference groups to have a fair comparison between them and add duplicate fusion genes to each of the potential groups it belongs. In the antisense case, it was done quick without distinguishing and just keeping the order above.

In [102]:
NLRs=pd.DataFrame()
for acc in ref["accession"].unique().tolist():
    
    clear_isos=ref.loc[(ref["accession"]==acc)&(ref["cat_assign"] == "one2one")]
    isos_type=augustus_refs(clear_isos).drop(columns=["ref_isoform","ref_gene"])
    nlr_isos=isos_type.loc[isos_type["type"]=="nlr"]

    
    b=fusion_types.loc[(fusion_types["nlr"]==1)&(fusion_types["accession"]==acc)]["isoform"].tolist()
    fusion_isos=ref.loc[(ref["accession"]==acc)&(ref["isoform"].isin(b))]

    c=pd.concat([nlr_isos,fusion_isos], ignore_index=True)
    
    c["isoforms_per_gene"]= c.groupby(by="gene")["isoform"].transform(lambda x: len(x.tolist()))
    c["ORFs_per_gene"]= c.groupby(by=["gene"])["Sqanti_ORF_seq"].transform(lambda x: len(set(x.tolist())))
    c["isoforms_per_ORFs"]= c.groupby(by=["gene","Sqanti_ORF_seq"])["isoform"].transform(lambda x: len(x.tolist()))
    
    #print("Mean nlr isoforms per gene:")
    hipg=c[["gene","isoforms_per_gene"]].drop_duplicates()
    #print(hipg["isoforms_per_gene"].apply("mean"))

    #print("Mean nlr ORFs per gene:")
    hopg=c[["gene","ORFs_per_gene"]].drop_duplicates()
    #print(hopg["ORFs_per_gene"].apply("mean"))
    #print("\n")
    
    
    NLRs=pd.concat([NLRs,c])
NLRs["type"]="nlr"

In [103]:
House=pd.DataFrame()
for acc in ref["accession"].unique().tolist():
    
    clear_isos=ref.loc[(ref["accession"]==acc)&(ref["cat_assign"] == "one2one")]
    isos_type=augustus_refs(clear_isos).drop(columns=["ref_isoform","ref_gene"])
    house_isos=isos_type.loc[isos_type["type"]=="housekeeping"]

    
    b=fusion_types.loc[(fusion_types["housekeeping"]==1)&(fusion_types["accession"]==acc)]["isoform"].tolist()
    fusion_isos=ref.loc[(ref["accession"]==acc)&(ref["isoform"].isin(b))]

    c=pd.concat([house_isos,fusion_isos], ignore_index=True)
    #c=ref.loc[(ref["accession"]==f"{acc}") & (ref["gene"].isin(set(a[f"{acc}"].tolist() + b[f"{acc}"].tolist())))&~(ref["cat_assign"].isin(["antisense","novel"]))].copy()
    
    c["isoforms_per_gene"]= c.groupby(by="gene")["isoform"].transform(lambda x: len(x.tolist()))
    c["ORFs_per_gene"]= c.groupby(by=["gene"])["Sqanti_ORF_seq"].transform(lambda x: len(set(x.tolist())))
    c["isoforms_per_ORFs"]= c.groupby(by=["gene","Sqanti_ORF_seq"])["isoform"].transform(lambda x: len(x.tolist()))
    
    #print("Mean nlr isoforms per gene:")
    hipg=c[["gene","isoforms_per_gene"]].drop_duplicates()
    #print(hipg["isoforms_per_gene"].apply("mean"))

    #print("Mean nlr ORFs per gene:")
    hopg=c[["gene","ORFs_per_gene"]].drop_duplicates()
    #print(hopg["ORFs_per_gene"].apply("mean"))
    #print("\n")
    
    
    House=pd.concat([House,c])
House["type"]="housekeeping"

In [104]:
Busco=pd.DataFrame()
for acc in ref["accession"].unique().tolist():
    
    clear_isos=ref.loc[(ref["accession"]==acc)&(ref["cat_assign"] == "one2one")]
    isos_type=augustus_refs(clear_isos).drop(columns=["ref_isoform","ref_gene"])
    busco_isos=isos_type.loc[isos_type["type"]=="busco"]

    
    b=fusion_types.loc[(fusion_types["busco"]==1)&(fusion_types["accession"]==acc)]["isoform"].tolist()
    fusion_isos=ref.loc[(ref["accession"]==acc)&(ref["isoform"].isin(b))]

    c=pd.concat([busco_isos,fusion_isos], ignore_index=True)
    #c=ref.loc[(ref["accession"]==f"{acc}") & (ref["gene"].isin(set(a[f"{acc}"].tolist() + b[f"{acc}"].tolist())))&~(ref["cat_assign"].isin(["antisense","novel"]))].copy()
    
    c["isoforms_per_gene"]= c.groupby(by="gene")["isoform"].transform(lambda x: len(x.tolist()))
    c["ORFs_per_gene"]= c.groupby(by=["gene"])["Sqanti_ORF_seq"].transform(lambda x: len(set(x.tolist())))
    c["isoforms_per_ORFs"]= c.groupby(by=["gene","Sqanti_ORF_seq"])["isoform"].transform(lambda x: len(x.tolist()))
    
    #print("Mean nlr isoforms per gene:")
    hipg=c[["gene","isoforms_per_gene"]].drop_duplicates()
    #print(hipg["isoforms_per_gene"].apply("mean"))

    #print("Mean nlr ORFs per gene:")
    hopg=c[["gene","ORFs_per_gene"]].drop_duplicates()
    #print(hopg["ORFs_per_gene"].apply("mean"))
    #print("\n")
    
    
    Busco=pd.concat([Busco,c])
Busco["type"]="busco"

In [105]:
Other=pd.DataFrame()
for acc in ref["accession"].unique().tolist():
    
    clear_isos=ref.loc[(ref["accession"]==acc)&(ref["cat_assign"] == "one2one")]
    isos_type=augustus_refs(clear_isos).drop(columns=["ref_isoform","ref_gene"])
    isos_type["type"] = isos_type["type"].fillna("other")
    other_isos=isos_type.loc[isos_type["type"]=="other"]

    
    b=fusion_types.loc[(fusion_types["other"]==1)&(fusion_types["accession"]==acc)]["isoform"].tolist()
    fusion_isos=ref.loc[(ref["accession"]==acc)&(ref["isoform"].isin(b))]

    c=pd.concat([other_isos,fusion_isos], ignore_index=True)
    
    c["isoforms_per_gene"]= c.groupby(by="gene")["isoform"].transform(lambda x: len(x.tolist()))
    c["ORFs_per_gene"]= c.groupby(by=["gene"])["Sqanti_ORF_seq"].transform(lambda x: len(set(x.tolist())))
    c["isoforms_per_ORFs"]= c.groupby(by=["gene","Sqanti_ORF_seq"])["isoform"].transform(lambda x: len(x.tolist()))
    
    #print("Mean nlr isoforms per gene:")
    hipg=c[["gene","isoforms_per_gene"]].drop_duplicates()
    #print(hipg["isoforms_per_gene"].apply("mean"))

    #print("Mean nlr ORFs per gene:")
    hopg=c[["gene","ORFs_per_gene"]].drop_duplicates()
    #print(hopg["ORFs_per_gene"].apply("mean"))
    #print("\n")
    
    
    Other=pd.concat([Other,c])
Other["type"]="other"

In [106]:
all_types=pd.concat([NLRs,Busco,House,Other], ignore_index=True)
all_types

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,subcategory,Sqanti_ORF_seq,all_canonical,cat_assign,association,type,isoforms_per_gene,ORFs_per_gene,isoforms_per_ORFs
0,G1128,G1128.10,at6923,at6923_G016360,at6923_G016360_mRNA01,1,3275,1,59,full-splice_match,mono-exon,MGNCVSLDVSCHHQTLNHACGCLFGDGNYIHMMEANLEALEKTIQE...,NaN,one2one,1,nlr,13,3,9.0
1,G1128,G1128.11,at6923,at6923_G016360,at6923_G016360_mRNA01,1,2744,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,NaN,one2one,1,nlr,13,3,3.0
2,G1128,G1128.12,at6923,at6923_G016360,at6923_G016360_mRNA01,1,2795,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,NaN,one2one,1,nlr,13,3,3.0
3,G1128,G1128.13,at6923,at6923_G016360,at6923_G016360_mRNA01,1,1821,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,NaN,one2one,1,nlr,13,3,3.0
4,G1128,G1128.2,at6923,at6923_G016360,novel,2,3331,1,59,genic,multi-exon,MGNCVSLDVSCHHQTLNHACGCLFGDGNYIHMMEANLEALEKTIQE...,canonical,one2one,1,nlr,13,3,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3262388,G16724,G16724.10,at9900,at9900_G337000_at9900_G336990,novel,5,2502,3,28,fusion,multi-exon,MAELKLSESRDLTRVERIGAHSHIRGLGLDSALEPRAVSEGMVGQV...,canonical,fusion,2,other,19,4,16.0
3262389,G16724,G16724.16,at9900,at9900_G337000_at9900_G336990,at9900_G336990_mRNA01,2,1810,1,28,full-splice_match,alternative_5end,MAELKLSESRDLTRVERIGAHSHIRGLGLDSALEPRAVSEGMVGQV...,canonical,fusion,2,other,19,4,16.0
3262390,G16724,G16724.17,at9900,at9900_G337000_at9900_G336990,novel,5,2482,1,28,fusion,multi-exon,MAELKLSESRDLTRVERIGAHSHIRGLGLDSALEPRAVSEGMVGQV...,canonical,fusion,2,other,19,4,16.0
3262391,G16724,G16724.19,at9900,at9900_G337000_at9900_G336990,novel,5,2381,1,28,fusion,multi-exon,MAELKLSESRDLTRVERIGAHSHIRGLGLDSALEPRAVSEGMVGQV...,canonical,fusion,2,other,19,4,16.0


In [107]:
drop=[]
for i in tqdm(all_types.groupby("accession")):
    df = i[1]
    boolean=df["isoform"].value_counts()>1
    boolean=boolean.rename_axis('isoform').reset_index(name='bool')
    #print(boolean.loc[boolean["bool"]==True])
    #print(boolean)
    dups = boolean.loc[boolean["bool"]==True]["isoform"].tolist()
    for iso in dups:
        iso_df=df[df["isoform"]==iso]
        if "nlr" in iso_df["type"].tolist():
            drop.extend(iso_df[~(iso_df["type"]=="nlr")].index.tolist())
            continue
        if "housekeeping" in iso_df["type"].tolist():
            drop.extend(iso_df[~(iso_df["type"]=="housekeeping")].index.tolist())
            continue
        if "busco" in iso_df["type"].tolist():
            drop.extend(iso_df[~(iso_df["type"]=="busco")].index.tolist())
            continue

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17/17 [01:32<00:00,  5.44s/it]


In [108]:
all_types=all_types.drop(drop).drop_duplicates()

In [109]:
all_types

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,subcategory,Sqanti_ORF_seq,all_canonical,cat_assign,association,type,isoforms_per_gene,ORFs_per_gene,isoforms_per_ORFs
0,G1128,G1128.10,at6923,at6923_G016360,at6923_G016360_mRNA01,1,3275,1,59,full-splice_match,mono-exon,MGNCVSLDVSCHHQTLNHACGCLFGDGNYIHMMEANLEALEKTIQE...,NaN,one2one,1,nlr,13,3,9.0
1,G1128,G1128.11,at6923,at6923_G016360,at6923_G016360_mRNA01,1,2744,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,NaN,one2one,1,nlr,13,3,3.0
2,G1128,G1128.12,at6923,at6923_G016360,at6923_G016360_mRNA01,1,2795,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,NaN,one2one,1,nlr,13,3,3.0
3,G1128,G1128.13,at6923,at6923_G016360,at6923_G016360_mRNA01,1,1821,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,NaN,one2one,1,nlr,13,3,3.0
4,G1128,G1128.2,at6923,at6923_G016360,novel,2,3331,1,59,genic,multi-exon,MGNCVSLDVSCHHQTLNHACGCLFGDGNYIHMMEANLEALEKTIQE...,canonical,one2one,1,nlr,13,3,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3262388,G16724,G16724.10,at9900,at9900_G337000_at9900_G336990,novel,5,2502,3,28,fusion,multi-exon,MAELKLSESRDLTRVERIGAHSHIRGLGLDSALEPRAVSEGMVGQV...,canonical,fusion,2,other,19,4,16.0
3262389,G16724,G16724.16,at9900,at9900_G337000_at9900_G336990,at9900_G336990_mRNA01,2,1810,1,28,full-splice_match,alternative_5end,MAELKLSESRDLTRVERIGAHSHIRGLGLDSALEPRAVSEGMVGQV...,canonical,fusion,2,other,19,4,16.0
3262390,G16724,G16724.17,at9900,at9900_G337000_at9900_G336990,novel,5,2482,1,28,fusion,multi-exon,MAELKLSESRDLTRVERIGAHSHIRGLGLDSALEPRAVSEGMVGQV...,canonical,fusion,2,other,19,4,16.0
3262391,G16724,G16724.19,at9900,at9900_G337000_at9900_G336990,novel,5,2381,1,28,fusion,multi-exon,MAELKLSESRDLTRVERIGAHSHIRGLGLDSALEPRAVSEGMVGQV...,canonical,fusion,2,other,19,4,16.0


In [110]:
# This is the fall_typesdataframe to continue 
all_types
all_types.to_pickle("output/01/clear-fusion_tama_nlrV2_hodgepodge.pkl")
#all_types.to_pickle("../5_NLRs/clear-fusion_tama_augustus.pkl")
#all_types.to_pickle("../3_fusion_genes/clear-fusion_tama_augustus.pkl")